# Experimento 5D: PubMedBERT + data augmentation con LLM, prompts corregidos (ALTERNATIVE_NAME)

**Objetivo:** repetir el experimento 5A/5B pero con los datos sinteticos de
`ALTERNATIVE_NAME` regenerados usando prompts corregidos, para comprobar si
la razon de que 5A/5B empeoraran era que **los datos generados estaban mal
etiquetados** (hipotesis a contrastar) en vez de -- o ademas de -- el
problema de confianza/threshold ya diagnosticado en la seccion 9 del informe.

**Que se corrigio respecto a la version original** (ver seccion 10 de
`DATA-AUGMENTATION-Y-FOCAL-LOSS-INFORME-FINAL.md`, auditoria manual completa
de los dos ficheros sinteticos):

- `augment_llm.py`: se encontraron 6/189 parafraseos (~3-4%) donde la frase
  generada usaba lenguaje de asociacion/progresion/coexistencia en vez de
  equivalencia (p.ej. "share pathophysiological features with" para un par
  que deberia decir "also known as"). Se añadio una regla explicita al
  prompt prohibiendo ese tipo de lenguaje para items `ALTERNATIVE_NAME`.
- `augment_llm_newpairs.py`: fallo sistematico, 20/20 frases no afirmaban
  realmente la equivalencia (solo usaban las dos palabras en clausulas
  cercanas del mismo organo/condicion, sin marcador explicito) -- causa
  encontrada en el propio prompt, que solo pedia "nearby clauses" en vez de
  exigir un marcador de equivalencia. Se corrigio para exigirlo
  explicitamente. Auditoria manual completa tras la regeneracion: 44-45/47
  con marcador correcto (antes 0/47).

**Datos:** `eng_train_plus_llmaug_v3.txt` = `eng_train.txt` (12739 lineas) +
`eng_train_llmaug_v2.txt` (190 lineas, parafraseo de ALTERNATIVE_NAME con
prompt corregido) + `eng_train_llmaug_newpairs_v2.txt` (47 lineas, parejas
nuevas con prompt corregido). Nota: a diferencia de 5A, aqui **no se incluye
`APPLIED_TO`** -- este experimento se centra solo en `ALTERNATIVE_NAME`,
que es la relacion auditada y corregida.

**Comparaciones relevantes ya disponibles** (mismo protocolo, seed 42):

| Experimento | Datos | ALTERNATIVE_NAME F1 (ciego calibrado) |
|---|---|---|
| 3A (baseline, sin augment) | -- | 0.131 |
| 5A (augment parafraseo, prompt SIN corregir) | 95 pares reales, 189 parafraseos | 0.111 |
| 5B (augment + parejas nuevas, prompt SIN corregir) | +20 parejas nuevas (0/20 con marcador) | 0.056 |
| **5D (este experimento)** | mismos pares, prompts CORREGIDOS | **?** |

Si 5D sigue igual o peor que 3A, la causa de fondo confirmada en la seccion 9
(problema de confianza/threshold, no de volumen ni calidad superficial de
las frases) queda reforzada. Si 5D mejora claramente sobre 5A/5B y se
acerca o supera a 3A, la calidad de los datos generados si era un factor
real que se sumaba al problema de fondo.

**Todo lo demas identico a 3A/5A/5B** (bugs de tokenizacion arreglados con
`fix_entity_markers()`, mismos hiperparametros, mismo protocolo de
evaluacion ciega calibrada, misma seed 42). Recordatorio: **una sola seed**,
no confirmar conclusiones sin repetir con mas seeds si el resultado es
prometedor.

## 1. Setup

In [ ]:
# Ejecucion en servidor local (zape), entorno conda "tfg". Mismo patron que 3A/5A/5B.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])

In [ ]:
import json, time, logging, gc, random
from collections import Counter
from pathlib import Path

import nltk, pandas as pd, torch, numpy as np

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric, fix_entity_markers
add_macro_f1_metric()
fix_entity_markers()   # identico a 3A/5A/5B -- ver HALLAZGOS-BUGS-TOKENIZACION.md
from score import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 2. Preparar el train aumentado (v2: prompts corregidos, solo ALTERNATIVE_NAME)

In [ ]:
DATA_DIR         = Path("../data/english")
ORIG_TRAIN       = DATA_DIR / "eng_train.txt"
LLMAUG_V2        = DATA_DIR / "eng_train_llmaug_v2.txt"          # parafraseo, prompt corregido
NEWPAIRS_V2      = DATA_DIR / "eng_train_llmaug_newpairs_v2.txt"  # parejas nuevas, prompt corregido
COMBINED_TRAIN   = DATA_DIR / "eng_train_plus_llmaug_v3.txt"

for p in (ORIG_TRAIN, LLMAUG_V2, NEWPAIRS_V2):
    assert p.exists(), f"FALTA {p} -- hay que regenerar con augment_llm.py / augment_llm_newpairs.py primero"

orig_lines  = [l for l in open(ORIG_TRAIN, encoding="utf-8") if l.strip()]
aug_lines   = [l for l in open(LLMAUG_V2, encoding="utf-8") if l.strip()]
new_lines   = [l for l in open(NEWPAIRS_V2, encoding="utf-8") if l.strip()]
all_aug_lines = aug_lines + new_lines

with open(COMBINED_TRAIN, "w", encoding="utf-8") as f:
    f.writelines(orig_lines)
    f.writelines(all_aug_lines)

orig_rel_counts = Counter(json.loads(l)["relation"] for l in orig_lines)
aug_rel_counts  = Counter(json.loads(l)["relation"] for l in all_aug_lines)
print(f"original: {len(orig_lines)} | aumentado (v2, corregido): {len(all_aug_lines)} "
      f"({len(aug_lines)} parafraseo + {len(new_lines)} parejas nuevas) | "
      f"combinado: {len(orig_lines)+len(all_aug_lines)}")
print(f"\n{'relacion':<20}{'original':>10}{'+aumentado':>12}{'total':>8}")
for rel in sorted(set(aug_rel_counts) | {"ALTERNATIVE_NAME"}):
    o, a = orig_rel_counts[rel], aug_rel_counts[rel]
    print(f"{rel:<20}{o:>10}{a:>12}{o+a:>8}")

## 3. Configuracion -- identica a 3A/5A/5B salvo el fichero de train

In [ ]:
MODEL_NAME      = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
EXPERIMENT_NAME = "pubmedbert_llmaug_v2fixed"
TECHNIQUE       = ("eng_train.txt + eng_train_llmaug_v2.txt + eng_train_llmaug_newpairs_v2.txt "
                    "concatenados (237 ejemplos LLM de ALTERNATIVE_NAME, generados con prompts "
                    "corregidos tras la auditoria de calidad de la seccion 10 del informe) -- "
                    "sobre fix_entity_markers(), unico cambio vs 3A")

MAX_LENGTH     = 256
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
EPOCHS         = 15
WARMUP_STEPS   = 300
SEED           = 42
GRAD_CLIP_NORM = 1.0

TRAIN_DATA  = COMBINED_TRAIN   # <-- el unico cambio real respecto a 3A
DEV_DATA    = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
id2rel = {v: k for k, v in rel2id.items()}
NO_REL_ID = rel2id["no_relation"]
print(f"Clases: {len(rel2id)} | modelo: {MODEL_NAME}")

OUT_DIR = Path(f"../outputs/5D-pubmedbert-llmaug-fixed/seed{SEED}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
print("Salida:", OUT_DIR)

## 4. Entrenamiento -- misma funcion que 1G/3A/5A/5B (`train_with_history`)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history

In [ ]:
set_seed(SEED)

encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL_NAME)
model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
framework = opennre.framework.SentenceRE(
    model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
    ckpt=str(CKPT_PATH), batch_size=BATCH_SIZE, max_epoch=EPOCHS, lr=LEARNING_RATE,
    opt="adamw", warmup_step=WARMUP_STEPS)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parametros: {n_params:,}")
print(f"Instancias de train: {len(orig_lines) + len(all_aug_lines)} "
      f"({len(orig_lines)} originales + {len(all_aug_lines)} aumentadas)")

t0 = time.time()
history = train_with_history(framework, EPOCHS, metric="macro_f1")
train_minutes = (time.time() - t0) / 60
with open(OUT_DIR / f"history_{EXPERIMENT_NAME}.json", "w") as f:
    json.dump(history, f, indent=2)
best = max(history, key=lambda h: h["val_macro_f1"])
macro_f1_curado = best["val_macro_f1"]
print(f"\nEntreno: {train_minutes:.1f} min | mejor epoch={best['epoch']} macro_f1_curado(dev)={macro_f1_curado:.4f}")

## 5. Inferencia en blind + evaluacion oficial (argmax)

In [ ]:
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

model.load_state_dict(torch.load(str(CKPT_PATH), map_location="cpu")["state_dict"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device); model.eval()

t0 = time.time()
all_probs = np.zeros((len(blind_raw), len(rel2id)), dtype=np.float32)
with torch.no_grad():
    for s in range(0, len(blind_raw), 64):
        batch = blind_raw[s:s + 64]
        tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                 "t": {"pos": i["t"]["pos"]}}) for i in batch]
        fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
        logits = model(*fields)
        all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
blind_minutes = (time.time() - t0) / 60
np.save(OUT_DIR / f"blind_probs_{EXPERIMENT_NAME}.npy", all_probs)
print(f"Inferencia blind: {blind_minutes:.1f} min")

def rows_from_preds(pred_ids):
    labels = [id2rel[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

argmax_ids = all_probs.argmax(axis=1)
res_argmax = evaluate(pd.DataFrame(rows_from_preds(argmax_ids)), gold_df_blind)
macro_f1_ciego_argmax = res_argmax["macro_f1"]
print(f"Macro F1 ciego (argmax puro): {macro_f1_ciego_argmax:.4f}")

## 6. Calibracion de threshold (grid fino) -- comparacion contra 3A, 5A y 5B

In [ ]:
def preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def eval_at(probs, threshold):
    pred_ids = preds_at_threshold(probs, threshold)
    return evaluate(pd.DataFrame(rows_from_preds(pred_ids)), gold_df_blind)

FINE_GRID = [round(float(x), 4) for x in np.arange(0.85, 0.9901, 0.001)] + \
            [round(float(x), 4) for x in np.arange(0.990, 0.9991, 0.001)] + \
            [0.9995, 0.9999]
FINE_GRID = sorted(set(FINE_GRID))

sweep = [(th, eval_at(all_probs, th)["macro_f1"]) for th in FINE_GRID]
best_threshold, macro_f1_ciego_calibrado = max(sweep, key=lambda x: x[1])
en_borde = best_threshold == FINE_GRID[-1]
per_relation_calibrado = eval_at(all_probs, best_threshold)["per_relation"]
print(f"Mejor threshold (fino): {best_threshold:.4f} -> Macro F1 ciego calibrado = {macro_f1_ciego_calibrado:.4f}"
      f"{'  [BORDE DEL GRID -- revisar]' if en_borde else ''}")

# --- 3A, 5A, 5B, cargados de los ficheros reales (no hardcodeados) ---
BASELINE_PATH = Path("../outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json")
FIVEA_PATH    = Path("../outputs/5A-pubmedbert-llmaug/seed42/results_seed_summary.json")
FIVEB_PATH    = Path("../outputs/5B-pubmedbert-llmaug-newpairs/seed42/results_seed_summary.json")
baseline_3a = json.load(open(BASELINE_PATH))
res_5a = json.load(open(FIVEA_PATH)) if FIVEA_PATH.exists() else None
res_5b = json.load(open(FIVEB_PATH)) if FIVEB_PATH.exists() else None

print(f"\n{'':<32}{'argmax':>10}{'calibrado':>12}{'ALT_NAME F1':>14}")
print(f"{'3A baseline (sin augment)':<32}{baseline_3a['macro_f1_ciego_argmax']:>10.4f}"
      f"{baseline_3a['macro_f1_ciego_calibrado']:>12.4f}"
      f"{baseline_3a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1', float('nan')):>14.4f}")
if res_5a:
    print(f"{'5A (augment, prompt SIN corregir)':<32}{res_5a['macro_f1_ciego_argmax']:>10.4f}"
          f"{res_5a['macro_f1_ciego_calibrado']:>12.4f}"
          f"{res_5a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1', float('nan')):>14.4f}")
if res_5b:
    print(f"{'5B (+parejas nuevas, SIN corregir)':<32}{res_5b['macro_f1_ciego_argmax']:>10.4f}"
          f"{res_5b['macro_f1_ciego_calibrado']:>12.4f}"
          f"{res_5b.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1', float('nan')):>14.4f}")
alt_f1_5d = per_relation_calibrado.get('ALTERNATIVE_NAME', {}).get('f1', float('nan'))
print(f"{'5D (este experimento, CORREGIDO)':<32}{macro_f1_ciego_argmax:>10.4f}{macro_f1_ciego_calibrado:>12.4f}{alt_f1_5d:>14.4f}")
print(f"\ndelta ALTERNATIVE_NAME vs 3A: {alt_f1_5d - baseline_3a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1', float('nan')):+.4f}")
if res_5a:
    print(f"delta ALTERNATIVE_NAME vs 5A (mismo experimento, prompt sin corregir): "
          f"{alt_f1_5d - res_5a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1', float('nan')):+.4f}")

## 7. Desglose completo por relacion

In [ ]:
rows = []
for rel, m in per_relation_calibrado.items():
    if rel == "no_relation":
        continue
    rows.append({"relation": rel, "f1_5D_llmaug_fixed": m["f1"], "precision": m["precision"],
                 "recall": m["recall"], "support": m["support"]})
df_5d = pd.DataFrame(rows).set_index("relation").sort_values("f1_5D_llmaug_fixed")
print(df_5d.round(3).to_string())

print("\n--- Foco en ALTERNATIVE_NAME, serie completa ---")
serie = {"3A (sin augment)": baseline_3a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1')}
if res_5a: serie["5A (augment, sin corregir)"] = res_5a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1')
if res_5b: serie["5B (+newpairs, sin corregir)"] = res_5b.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1')
serie["5D (este experimento, corregido)"] = alt_f1_5d
for k, v in serie.items():
    print(f"  {k:<38}{v:.4f}" if v is not None else f"  {k:<38}N/A")

## 8. Guardar resultados

In [ ]:
results = {
    "exp": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "technique": TECHNIQUE,
    "seed": SEED,
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "warmup_steps": WARMUP_STEPS, "neg_ratio": 3, "seed": SEED,
    },
    "n_train_original": len(orig_lines),
    "n_train_llmaug": len(all_aug_lines),
    "n_train_llmaug_paraphrase": len(aug_lines),
    "n_train_llmaug_newpairs": len(new_lines),
    "n_train_total": len(orig_lines) + len(all_aug_lines),
    "macro_f1_curado": macro_f1_curado,
    "macro_f1_ciego_argmax": macro_f1_ciego_argmax,
    "best_threshold_fino": best_threshold,
    "macro_f1_ciego_calibrado": macro_f1_ciego_calibrado,
    "en_borde_del_grid_fino": en_borde,
    "per_relation_ciego_calibrado": per_relation_calibrado,
    "train_minutes": round(train_minutes, 1),
    "blind_minutes": round(blind_minutes, 1),
    "baseline_comparison": {
        "source_3A": "outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json (sin augment)",
        "source_5A": "outputs/5A-pubmedbert-llmaug/seed42/results_seed_summary.json (augment, prompt SIN corregir)",
        "source_5B": "outputs/5B-pubmedbert-llmaug-newpairs/seed42/results_seed_summary.json (+newpairs, SIN corregir)",
        "baseline_3a_macro_f1_ciego_calibrado": baseline_3a["macro_f1_ciego_calibrado"],
        "delta_vs_3a_calibrado": macro_f1_ciego_calibrado - baseline_3a["macro_f1_ciego_calibrado"],
        "delta_vs_3a_alternative_name": alt_f1_5d - baseline_3a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1', float('nan')),
        "delta_vs_5a_alternative_name": (alt_f1_5d - res_5a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1', float('nan'))) if res_5a else None,
    },
}
with open(OUT_DIR / "results_seed_summary.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Guardado:", OUT_DIR / "results_seed_summary.json")
baseline_3a_alt_f1 = baseline_3a.get('per_relation_ciego_calibrado', {}).get('ALTERNATIVE_NAME', {}).get('f1')
if baseline_3a_alt_f1 is not None:
    print(f"\nALTERNATIVE_NAME F1 ciego calibrado: 3A={baseline_3a_alt_f1:.4f} -> 5D(corregido)={alt_f1_5d:.4f}")
else:
    print(f"\nALTERNATIVE_NAME F1 ciego calibrado (5D, corregido): {alt_f1_5d:.4f} "
          f"(3A no guarda desglose por relacion en su propio json -- ver informe seccion 7 para 3A=0.131)")

## 9. Conclusion (rellenar tras ver los resultados)

Recordatorio antes de sacar conclusiones -- **esto es una sola seed**. El
estudio multiseed (1G) midio std~0.008-0.017 entre seeds para este mismo
encoder en ciego calibrado, asi que un delta menor que ~2-3x eso no se puede
afirmar como real todavia sin repetir con mas seeds (ver
`methodology_multiseed_comparisons` en la memoria del proyecto).

**Pregunta que responde este experimento:** ¿la mala calidad de los datos
sinteticos (deriva semantica en 5A, fallo sistematico de marcador en 5B,
ambos corregidos aqui) explica el resultado negativo, o el problema de
fondo (confianza/threshold, seccion 9 del informe) persiste incluso con
datos limpios?

- Si `ALTERNATIVE_NAME` en 5D se acerca o supera a 3A (0.131) y mejora
  claramente sobre 5A (0.111) y 5B (0.056): la calidad de los datos SI era
  un factor real, ademas del problema de threshold. Vale la pena documentar
  ambos hallazgos juntos en la memoria.
- Si 5D sigue en el rango de 5A/5B (0.05-0.11) a pesar de la calidad
  corregida: la explicacion de la seccion 9 (el modelo distingue "reconoce
  con seguridad" vs "no reconoce en absoluto", y mas ejemplos -- limpios o
  no -- no mueven esa frontera) queda reforzada como la causa dominante, y
  la conclusion final del informe (focal loss > cualquier variante de
  augmentation) se mantiene sin cambios.